<a id="top"></a>

# Lab L1106: build a routing workflow (and swap the decider)

```yaml
title: "Lab L1106: build a routing workflow (and swap the decider)"
keywords: langgraph, routing, conditional edge, classifier, user-input branching, deterministic, eval, workflow vs agent, lab
estimated duration: 45
```

> **Lesson:** L11. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L11/objectives.md).
> **This is the solutions key** for `L1106_lab_empty.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` stands in for `ChatAnthropic`. No API key
> needed. Builds on [Lab L1104](L1104_lab_empty.ipynb).

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- Routing state + classify node](#2-problem-1----routing-state--classify-node)
- [3. Problem 2 -- The branch nodes](#3-problem-2----the-branch-nodes)
- [4. Problem 3 -- The conditional edge](#4-problem-3----the-conditional-edge)
- [5. Problem 4 -- Run and prove determinism](#5-problem-4----run-and-prove-determinism)
- [6. Problem 5 -- Same graph, user-input branch](#6-problem-5----same-graph-user-input-branch)
- [7. Problem 6 (optional) -- Evaluate the classifier](#7-problem-6-optional----evaluate-the-classifier)
- [8. Problem 7 -- Workflow vs. agent (written)](#8-problem-7----workflow-vs-agent-written)
- [9. Self-check](#9-self-check)

## 1. Setup

Given again: the offline `StubChat` (this one also returns a **category** when asked to classify),
the sample `TICKETS`, and the `haiku`/`sonnet` stub clients. Run this first. (Same setup as Lab
L1104 -- this lab stands alone.)

In [1]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, StateGraph

HAIKU = "claude-haiku-4-5"  # the names are real; here a stub stands in for the client
SONNET = "claude-sonnet-4-6"

TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
    "general": "Do you offer a student discount, and how would I apply it to my account?",
}
POLICY = (
    "Refunds: a duplicate charge is always refundable within 60 days. "
    "Never promise a refund for change-of-mind. "
    "Escalate anything mentioning legal action or chargebacks to a human."
)


class StubReply:
    """Mimics the one field we read off a ChatAnthropic response: `.content`."""

    def __init__(self, content: str) -> None:
        self.content = content


class StubChat:
    """A tiny OFFLINE stand-in for ChatAnthropic, so this lab is free, fast, and deterministic.

    It mimics the single call our nodes use -- `.invoke(prompt).content` -> str. The graph
    wiring you practice here is identical with a real model: swapping `StubChat(HAIKU)` for
    `ChatAnthropic(model=HAIKU, api_key=require_anthropic_key())` is a one-line change, and the
    node code never changes (Problem 5). The lecture demos use the real client.
    """

    def __init__(self, model: str) -> None:
        self.model = model

    def invoke(self, prompt: str) -> StubReply:
        p = prompt.lower()
        if "classify" in p:
            # Key on TICKET-content words only. We avoid "bill"/"technical"/"general" because
            # those leak from the instruction text itself ("billing, technical, or general").
            if any(w in p for w in ("charge", "charged", "refund", "invoice", "payment", "twice")):
                return StubReply("billing")
            if any(w in p for w in ("error", "500", "crash", "bug", "broken", "login", "log in")):
                return StubReply("technical")
            return StubReply("general")
        if "summarize" in p:
            return StubReply("The customer needs help resolving an account issue.")
        if "compliance" in p or "policy" in p:
            return StubReply("OK: the reply is consistent with the refund policy.")
        return StubReply("Thanks for reaching out -- happy to help you with this!")


# Per-node models -- a stub each, exactly where a real graph would bind Haiku vs. Sonnet.
haiku = StubChat(HAIKU)
sonnet = StubChat(SONNET)

[↑ Back to top](#top)

## 2. Problem 1 -- Routing state + classify node

Define `RouteState` (it needs `ticket`, `category`, `user_choice`, `draft`, and the `steps` append
list), then write the `classify` entry node. `classify` runs on the **cheap** stub (`haiku`) because
it only produces a one-word label; keep only `billing`/`technical`/`general` and default to
`general`.

In [2]:
class RouteState(TypedDict):
    """State for the routing workflow.

    `category` is set by the classify node (a model label); `user_choice` is set by the USER in
    the initial state (Problem 5). One state type carries both deciders.
    """

    ticket: str
    category: str  # the classifier's label
    user_choice: str  # a path the user supplies directly
    draft: str
    steps: Annotated[list[str], add]


def classify(state: RouteState) -> dict[str, object]:
    """Label the ticket. A LIGHT step -> Haiku (we only need one word)."""
    reply = haiku.invoke(
        "Classify this support ticket as exactly one word -- billing, technical, or general.\n\n"
        f"{state['ticket']}"
    )
    label = str(reply.content).strip().lower()
    category = next((c for c in ("billing", "technical", "general") if c in label), "general")
    return {"category": category, "steps": [f"classify->{category}"]}

[↑ Back to top](#top)

## 3. Problem 2 -- The branch nodes

Write the three specialized branch nodes -- `billing`, `technical`, `general`. Each runs on the
**capable** stub (`sonnet`), drafts a reply for `state["ticket"]`, and appends its name to `steps`.
The model works *inside* each node.

In [3]:
def billing(state: RouteState) -> dict[str, object]:
    reply = sonnet.invoke(f"Write a billing-support reply.\n\nTicket: {state['ticket']}")
    return {"draft": str(reply.content), "steps": ["billing"]}


def technical(state: RouteState) -> dict[str, object]:
    reply = sonnet.invoke(f"Write a technical-support reply.\n\nTicket: {state['ticket']}")
    return {"draft": str(reply.content), "steps": ["technical"]}


def general(state: RouteState) -> dict[str, object]:
    reply = sonnet.invoke(f"Write a friendly general-support reply.\n\nTicket: {state['ticket']}")
    return {"draft": str(reply.content), "steps": ["general"]}

[↑ Back to top](#top)

## 4. Problem 3 -- The conditional edge

Write `route(state)` returning `state["category"]`, then wire it with `add_conditional_edges`. Say
the line as you go: **this is not the model deciding.** The model produced a *label*; your routing
function reads that label and picks the edge. (In L12 the routing function will branch on whether the
*model asked for a tool* -- same mechanism, different decider.) Compile to `route_app` and print the
diagram -- it now **branches**.

In [4]:
def route(state: RouteState) -> str:
    """Pick the branch from the label ALREADY in state -- no model call in this decision."""
    return state["category"]


builder = StateGraph(RouteState)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("general", general)
builder.set_entry_point("classify")
builder.add_conditional_edges(
    "classify",
    route,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
for branch_name in ("billing", "technical", "general"):
    builder.add_edge(branch_name, END)
route_app = builder.compile()

# The diagram now BRANCHES (dashed conditional edges) yet still always reaches END.
print(route_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	billing(billing)
	technical(technical)
	general(general)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> billing;
	classify -.-> general;
	classify -.-> technical;
	billing --> __end__;
	general --> __end__;
	technical --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[↑ Back to top](#top)

## 5. Problem 4 -- Run and prove determinism

Invoke once per category, then run the *same* ticket twice and show the path is identical. The
branch you land in is fixed by the classifier's label -- re-running never changes the path.

In [5]:
for kind in ("billing", "technical", "general"):
    out = route_app.invoke({"ticket": TICKETS[kind], "steps": []})
    print(f"{kind:9s} -> path {out['steps']}")

path_a = route_app.invoke({"ticket": TICKETS["billing"], "steps": []})["steps"]
path_b = route_app.invoke({"ticket": TICKETS["billing"], "steps": []})["steps"]
print("\nsame input, same path:", path_a == path_b, path_a)

billing   -> path ['classify->billing', 'billing']
technical -> path ['classify->technical', 'technical']
general   -> path ['classify->general', 'general']

same input, same path: True ['classify->billing', 'billing']


[↑ Back to top](#top)

## 6. Problem 5 -- Same graph, user-input branch

Now swap the **decider**. Replace `classify` with an `intake` pass-through node and a
`route_by_user` function that reads `state["user_choice"]` -- a value the **user** supplied in the
initial state. **No model is involved in the routing decision at all.** Feed a *technical* ticket but
set `user_choice="billing"`: the path must follow the **user**, not the ticket -- *the user owns the
edge, the model still works inside the chosen branch node.*

In [6]:
def intake(state: RouteState) -> dict[str, object]:
    """A plain pass-through node -- no model. The user has already chosen the path."""
    return {"steps": ["intake"]}


def route_by_user(state: RouteState) -> str:
    """Branch on the USER's choice from the initial state. No model in this decision."""
    return state["user_choice"]


ub = StateGraph(RouteState)
ub.add_node("intake", intake)
ub.add_node("billing", billing)
ub.add_node("technical", technical)
ub.add_node("general", general)
ub.set_entry_point("intake")
ub.add_conditional_edges(
    "intake",
    route_by_user,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
for branch_name in ("billing", "technical", "general"):
    ub.add_edge(branch_name, END)
user_app = ub.compile()

# Technical ticket, but the USER chose billing -> the path follows the user, not the content.
out = user_app.invoke({"ticket": TICKETS["technical"], "user_choice": "billing", "steps": []})
print("user chose billing -> path", out["steps"])

user chose billing -> path ['intake', 'billing']


[↑ Back to top](#top)

## 7. Problem 6 (optional) -- Evaluate the classifier

Carry the **L09 eval discipline** onto your workflow -- it's the cheapest thing to test, because the
same input always takes the same path. Run the small `EVAL_CASES` set through your `classify` node
and report the pass rate. One case is ambiguous on purpose; let the eval *surface* the weakness
rather than hide it. (In L09 this lived in a reusable harness; here it's inlined.)

In [7]:
# OPTIONAL. A deterministic workflow is the easiest thing to evaluate: same input -> same path.
# This is the L09 discipline, inlined: labeled inputs in, expected label out, count the passes.
EVAL_CASES = [
    ("My card was charged twice for one order.", "billing"),
    ("The dashboard shows a 500 error on load.", "technical"),
    ("What are your support hours?", "general"),
    ("The app keeps crashing and I want a refund.", "technical"),  # ambiguous on purpose
]

passed = 0
for ticket, expected in EVAL_CASES:
    got = classify({"ticket": ticket, "steps": []})["category"]
    ok = got == expected
    passed += int(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] expected {expected:9s} got {got!s:9s} :: {ticket}")
print(f"\npass rate: {passed}/{len(EVAL_CASES)}")
# The last case fails: it mentions a refund (billing) AND a crash (technical), and our classifier
# checks billing keywords first. That's exactly the kind of weakness an eval is meant to surface.

[PASS] expected billing   got billing   :: My card was charged twice for one order.
[PASS] expected technical got technical :: The dashboard shows a 500 error on load.
[PASS] expected general   got general   :: What are your support hours?
[FAIL] expected technical got billing   :: The app keeps crashing and I want a refund.

pass rate: 3/4


[↑ Back to top](#top)

## 8. Problem 7 -- Workflow vs. agent (written)

Both graphs you built are **workflows**: the developer wired every edge, and re-running takes the
same path. In a sentence or two: **what single change would turn one of them into an agent**, and
who would then control the path? (Hint: think about a *back-edge* and who decides whether to take
it. This is exactly the line into L12.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 9. Self-check

Compare against `L1106_lab_solutions.ipynb`. You're done when: `route_app` classifies then routes to
one branch and always reaches `END`; the same ticket takes the same path twice; the user-input graph
routes on `user_choice` with **no model in the decision** (a technical ticket with
`user_choice="billing"` lands in `billing`); the optional eval reports a pass rate with the ambiguous
case failing; and you can name the single change (a model-driven back-edge) that makes a workflow an
agent.

[↑ Back to top](#top)